# 1. テスト - Dockerコンテナ上のPostgreSQLへのアクセス

予め、DockerのPostgreSQL用コンテナを起動しておく必要があります。

## 1.1 Dockerコンテナ上のPostgreSQL - 接続

In [ ]:
import psycopg
from psycopg import OperationalError

# Dockerコンテナ内のPostgreSQLサーバに接続するためのパラメータ
connection_params = {
    "host": "127.0.0.1",
    "port": "5432",
    "user": "postgres",
    "password": "secret",
    "dbname": "postgres"  # デフォルトのデータベース名
}

print("PostgreSQLサーバへの接続を試みます...")

try:
    # connect() で接続を確立します
    # autocommit=True にしておくと、後で手動commitが不要になりテスト時に便利です
    with psycopg.connect(**connection_params, autocommit=True) as conn:
        
        # 接続情報の確認
        print(f"接続成功！")
        print(f"Backend PID: {conn.info.backend_pid}")
        
        # 念のため、簡単なSQLを実行して応答を確認します
        with conn.cursor() as cur:
            cur.execute("SELECT version();")
            db_version = cur.fetchone()
            print(f"Database Version: {db_version[0]}")

except OperationalError as e:
    print(f"接続失敗...: {e}")

## 1.2 Dockerコンテナ上のPostgreSQL - テーブル作成

In [ ]:
import psycopg
from psycopg import OperationalError

# Dockerコンテナ内のPostgreSQLサーバに接続するためのパラメータ
connection_params = {
    "host": "127.0.0.1",
    "port": "5432",
    "user": "postgres",
    "password": "secret",
    "dbname": "postgres"  # デフォルトのデータベース名
}

# テーブル作成用SQL (PostgreSQLの文法)
create_table_sql = """
CREATE TABLE IF NOT EXISTS mds_plant (
    id SERIAL PRIMARY KEY,
    name TEXT,
    description TEXT,
    mass_kg DOUBLE PRECISION, 
    damper_Ns_m DOUBLE PRECISION,
    spring_N_m DOUBLE PRECISION,
    spring_balance_pos_m DOUBLE PRECISION,
    static_friction_coeff DOUBLE PRECISION,
    dynamic_friction_coeff DOUBLE PRECISION,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
"""

try:
    # connect() で接続を確立します
    # autocommit=True にしておくと、後で手動commitが不要になりテスト時に便利です
    with psycopg.connect(**connection_params, autocommit=True) as conn:

        # 接続情報の確認
        print(f"接続成功！")
        print(f"Backend PID: {conn.info.backend_pid}")

        with conn.cursor() as cur:
            # SQL実行
            cur.execute(create_table_sql)
            # 変更を確定 (コミット) ※autocommit=Trueにしていない場合は必須
            conn.commit()
            print("テーブル 'mds_plant' の作成に成功しました（または既に存在します）。")
except OperationalError as e:
    print(f"接続失敗...: {e}")
except Exception as e:
    print(f"テーブル作成エラー: {e}")

## 1.3 Dockerコンテナ上のPostgreSQL - テーブル存在確認

In [ ]:
import psycopg

# 接続パラメータ（これまでと同じものです）
connection_params = {
    "host": "127.0.0.1",
    "port": "5432",
    "user": "postgres",
    "password": "secret",
    "dbname": "postgres"
}

def show_table_schema(table_name: str):
    # information_schema.columns から列の定義情報を取得するSQL
    sql = """
    SELECT 
        column_name, 
        data_type, 
        column_default, 
        is_nullable
    FROM 
        information_schema.columns
    WHERE 
        table_name = %s
    ORDER BY 
        ordinal_position;
    """

    try:
        with psycopg.connect(**connection_params) as conn:
            with conn.cursor() as cur:
                # プレースホルダを使ってテーブル名を渡します
                cur.execute(sql, (table_name,))
                columns = cur.fetchall()

                if not columns:
                    print(f"テーブル '{table_name}' が見つかりません。")
                    return

                # 結果を見やすく整形して出力
                print(f"=== テーブル '{table_name}' のスキーマ定義 ===")
                print(f"{'列名 (Column Name)':<25} | {'データ型 (Data Type)':<20} | {'デフォルト値 (Default)':<30} | {'NULL許可'}")
                print("-" * 100)
                
                for col in columns:
                    col_name = col[0]
                    data_type = col[1]
                    # デフォルト値が設定されていない場合は 'None' と表示
                    col_default = str(col[2]) if col[2] is not None else "None"
                    is_nullable = col[3]
                    
                    print(f"{col_name:<25} | {data_type:<20} | {col_default:<30} | {is_nullable}")

    except Exception as e:
        print(f"スキーマ取得エラー: {e}")

# 作成した 'mds_plant' テーブルの定義を確認
show_table_schema('mds_plant')

## 1.4 Dockerコンテナ上のPostgreSQL - テストデータの登録

In [ ]:
# 登録するテストデータ
plant_data = (
    "default_plant",          # name
    "Standard MDS parameters",# description
    1.0,                      # mass_kg
    0.5,                      # damper_Ns_m
    10.0,                     # spring_N_m
    0.0,                      # spring_balance_pos_m
    0.1,                      # static_friction_coeff
    0.05                      # dynamic_friction_coeff
)

insert_sql = """
INSERT INTO mds_plant (name, description, mass_kg, damper_Ns_m, spring_N_m, spring_balance_pos_m, static_friction_coeff, dynamic_friction_coeff)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
RETURNING id;
"""

try:
    with psycopg.connect(**connection_params, autocommit=True) as conn:
        with conn.cursor() as cur:
            # データの挿入（プレースホルダ %s を使用して安全に値を渡します）
            cur.execute(insert_sql, plant_data)
            inserted_id = cur.fetchone()[0]
            print(f"テストデータの登録に成功しました！ (ID: {inserted_id})")
except Exception as e:
    print(f"データ登録エラー: {e}")

## 1.5 Dockerコンテナ上のPostgreSQL - テストデータの読み込み

In [ ]:
def fetch_plant_params(plant_id: int) -> dict | None:
    select_sql = """
    SELECT mass_kg, damper_Ns_m, spring_N_m, spring_balance_pos_m, 
           static_friction_coeff, dynamic_friction_coeff
    FROM mds_plant
    WHERE id = %s;
    """
    
    try:
        with psycopg.connect(**connection_params) as conn:
            with conn.cursor() as cur:
                cur.execute(select_sql, (plant_id,))
                record = cur.fetchone()
                
                if record:
                    # 取得したタプルデータを辞書型に変換
                    return {
                        "mass_kg": record[0],
                        "damper_Ns_m": record[1],
                        "spring_N_m": record[2],
                        "spring_balance_pos_m": record[3],
                        "static_friction_coeff": record[4],
                        "dynamic_friction_coeff": record[5]
                    }
                else:
                    print("指定されたIDのデータが見つかりません。")
                    return None
                    
    except Exception as e:
        print(f"データ読み込みエラー: {e}")
        return None

# 実行テスト（Step 1で発行されたIDを指定してください）
params = fetch_plant_params(1)
print(params)

# 2. テスト - DBAccessor

## 2.1 DB接続

In [ ]:
from tkmotion.db.db_access import DBAccessor

dba = DBAccessor()
dba.connect()

## 2.2 テーブルの存在確認

In [ ]:
# 作成した 'mds_plant' テーブルの定義を確認
from tkmotion.db.db_access import DBAccessor

dba = DBAccessor()
dba.show_table_schema('mds_plant')

## 2.3 テストデータの読み込み

In [ ]:
# 実行テスト（Step 1で発行されたIDを指定してください）
from tkmotion.db.db_access import DBAccessor

dba = DBAccessor()
params = dba.fetch_plant_params(1)
print(params)

# 3. テスト - DBAccessorからのPlant読み込み

## 3.1 Plant読み込み

In [ ]:
import pprint
import tkmotion as tkm
flow = tkm.MotionFlow()
flow.load_plant_from_db()

pprint.pprint(type(flow.plant))
pprint.pprint(type(flow.plant.physical_obj))

pprint.pprint(flow.plant.get_config())

## 3.2 モーションフロー実行

In [ ]:
# discrete_time
flow.load_discrete_time()
# motion_profile
trapezoid_prof_index = 1
flow.load_motion_profile(filepath="tkmotion/prof/default_motion_prof_config.json", prof_index=trapezoid_prof_index)
# controller
pid_ctrl_index = 1
flow.load_controller(filepath="tkmotion/ctrl/default_controller_config.json", ctrl_index=pid_ctrl_index)

In [ ]:
# execute
df = flow.execute()
df

## 3.3 結果表示

In [ ]:
import plotly.express as px
fig = px.line(
    df,
    x="time_s",
    y=["cmd_velocity_m_s", "cmd_position_m"],
    title="Motion Profile Execution Result")
fig.update_layout(
    yaxis2=dict(
        title="Position (m)",  # 第2Y軸のタイトル
        overlaying="y",        # 第1Y軸の上に重ねる
        side="right",          # 右側に表示
        showgrid=False         # オプション: グリッド線を非表示
    ),
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)
for trace in fig.data:
    # 第2Y軸のトレース更新 / Update 2nd Y-axis trace
    if trace.name == "cmd_position_m":
        trace.update(yaxis="y2")
        break
fig.update_yaxes(title_text="Velocity (m/s)", row=1, col=1)
fig.update_xaxes(title_text="Time (s)", row=1, col=1)
fig.show()

In [ ]:
import plotly.express as px
fig = px.line(df, x="time_s", y=["velocity_error_m_s"], title="Velocity Error")
fig.update_layout(
    xaxis=dict(title="time [s]"),
    yaxis=dict(title="velocity error [m/s]"),
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.05))
fig.show()

In [ ]:
import plotly.express as px
fig = px.line(df, x="time_s", y=["position_error_m"], title="Position Error")
fig.update_layout(
    xaxis=dict(title="time [s]"),
    yaxis=dict(title="position error [m]"),
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.7))
fig.show()